In [6]:
from datetime import timedelta
import warnings
from pathlib import Path
import numpy as np
import pickle
import urllib.request
from io import StringIO
import pandas as pd
from glob import glob

import speasy as spz
from speasy.core import AnyDateTimeType, make_utc_datetime
from speasy.products.variable import merge
from speasy.products import SpeasyVariable, Dataset, DataContainer, VariableTimeAxis
from speasy.signal.resampling import resample as spz_resample, generate_time_vector


In [ ]:
#https://www.bcmt.fr/DATABANK/VARIATION/tam/min/2025/tam20250101vmin.min

def _load_ground_mag(url:str):
        
    with urllib.request.urlopen(url) as resp:
        lines = resp.read().decode("utf-8", errors="replace").splitlines()
    
    # Locate the data column-header line (IAGA-2002 header is variable length)
    hdr = next(i for i, ln in enumerate(lines) if ln.startswith("DATE"))
    cols = lines[hdr].split()
    if cols[-1] == "|":           # drop the trailing "|"
        cols = cols[:-1]
    
    df = pd.read_csv(
        StringIO("\n".join(lines[hdr + 1:])),
        sep=r"\s+",
        header=None,
        names=cols,               # DATE TIME DOY TAMX TAMY TAMZ TAMF
    )
    
    # Build a datetime index and clean up
    df["DATETIME"] = pd.to_datetime(df["DATE"] + " " + df["TIME"])
    df = df.set_index("DATETIME").drop(columns=["DATE", "TIME", "DOY"])
    
    # IAGA-2002 fill values -> NaN (99999 for components, 88888 for missing F)
    df = df.replace({99999.00: np.nan, 88888.00: np.nan})
    return spz.SpeasyVariable(
        axes=[VariableTimeAxis(np.array([np.datetime64(int(d.timestamp() * 1e9), "ns") for d in df.index]))], 
        values=DataContainer(df.values), 
        columns=list(df.columns)
    )


def _bcmt_url(station: str, day) -> str:
    s = station.lower()
    return (f"https://www.bcmt.fr/DATABANK/VARIATION/{s}/min/"
            f"{day:%Y}/{s}{day:%Y%m%d}vmin.min")

def load_ground_mag(start: AnyDateTimeType, stop: AnyDateTimeType, station: str):
    start = make_utc_datetime(start)
    stop = make_utc_datetime(stop)

    # One file per UTC day; include the day containing `stop`
    d0, d1 = start.date(), stop.date()
    days = [d0 + timedelta(days=i) for i in range((d1 - d0).days + 1)]

    variables = []
    for day in days:
        url = _bcmt_url(station, day)
        try:
            variables.append(_load_ground_mag(url))
        except Exception as e:              # missing day / 404 / parse error
            warnings.warn(f"skipping {url}: {e}")

    if not variables:
        return None

    merged = merge(variables)               # concatenates + sorts on time
    return merged[start:stop]               # half-open [start, stop)


In [ ]:
def flatten_values(values: list[SpeasyVariable or Dataset] or SpeasyVariable or Dataset) -> list[SpeasyVariable]:
    result = []
    if isinstance(values, (SpeasyVariable, Dataset)):
        values = [values]
    for value in values:
        if isinstance(value, SpeasyVariable):
            result.append(value)
        elif isinstance(value, Dataset):
            for var in value.variables.values():
                result.append(var)
    return result


def omni_data(start_date: AnyDateTimeType, stop_date: AnyDateTimeType) -> Dataset:
    variables = {}
    for uid, param in (
        ("cda/OMNI_HRO2_1MIN/proton_density", "ni"),
        ("cda/OMNI_HRO2_1MIN/Pressure", "Pdyn"),
        ("cda/OMNI_HRO2_1MIN/T", "T"),
        ("cda/OMNI_HRO2_1MIN/Vx", "Vx"),
        ("cda/OMNI_HRO2_1MIN/Vy", "Vy"),
        ("cda/OMNI_HRO2_1MIN/Vz", "Vz"),
        ("cda/OMNI_HRO2_1MIN/BX_GSE", "BX_GSE"),
        ("cda/OMNI_HRO2_1MIN/BY_GSE", "BY_GSE"),
        ("cda/OMNI_HRO2_1MIN/BZ_GSE", "BZ_GSE"),
    ):
        variables[param] = spz.get_data(uid, start_date, stop_date)
    return Dataset(name="OMNI", variables=variables, meta={})


def themis_data(start_date: AnyDateTimeType, stop_date: AnyDateTimeType, spacecraft='a') -> Dataset:
    variables = {}
    for uid, param in (
        (f"amda/th{spacecraft}_bs_gsm", f"th{spacecraft}_fgs_gse"),
        (f"amda/th{spacecraft}_v_i",    f"th{spacecraft}_Vi_gse"),
        (f"amda/th{spacecraft}_n_i",    f"th{spacecraft}_ni"),
        
    ):
        r = spz.get_data(uid, start_date, stop_date)
        if r is None:
            print(f"Oops got None with {param}, ({uid})")
        variables[param] = r
    return Dataset(name="THEMIS", variables=variables, meta={})


def mms1_data(start_date: AnyDateTimeType, stop_date: AnyDateTimeType) -> Dataset:
    variables = {}
    for uid, param in (
        ("amda/mms1_b_gse", "mms1_b_gse"),
        ("amda/mms1_dis_vgse", "mms1_dis_vgse"),
        ("amda/mms1_dis_ni", "mms1_dis_ni"),
    ):
        r = spz.get_data(uid, start_date, stop_date)
        if r is None:
            print(f"Oops got None with {param}, ({uid})")
        variables[param] = r
    return Dataset(name="MMS1", variables=variables, meta={})

def ground_mag_data(start_date: AnyDateTimeType, stop_date: AnyDateTimeType) -> Dataset:
    variables = {}
    for station in (
        "TAM", "SOK", "EDA", "CLF", "KOU", "IPM", "PPT"
    ):
        r = load_ground_mag(start_date, stop_date, station)
        if r is None:
            print(f"Oops got None with {station}")
        variables[station] = r
    return Dataset(name="Ground Stations MAG", variables=variables, meta={})

In [ ]:
omni_data("2025/01/01", "2025/02/01").plot()

In [ ]:
mms1_data("2025/01/01", "2025/02/01").plot()

In [ ]:
ground_mag_data("2025/01/01", "2025/02/01").plot()

In [ ]:
def _resample_variable(variable: SpeasyVariable, interval: float or np.timedelta64) -> SpeasyVariable:
    return spz_resample(variable, interval)

def _resample_dataset(dataset: Dataset, interval: float or np.timedelta64) -> Dataset:
    resampled_variables = {}
    for name, var in dataset.variables.items():
        resampled_variables[name] = _resample_variable(var, interval)
    return Dataset(name=dataset.name, variables=resampled_variables, meta=dataset.meta)

def resample(
    values: list[SpeasyVariable or Dataset] or SpeasyVariable or Dataset,
    interval: float or np.timedelta64,
) -> list[SpeasyVariable or Dataset] or SpeasyVariable or Dataset:
    if isinstance(values, SpeasyVariable) :
        return _resample_variable(values, interval)
    elif isinstance(values, Dataset):
        return _resample_dataset(values, interval)
    result = []
    for value in values:
        if isinstance(value, SpeasyVariable):
            result.append(_resample_variable(value, interval))
        elif isinstance(value, Dataset):
            result.append(_resample_dataset(value, interval))
    return result

In [ ]:
def save_dataset(values: list[SpeasyVariable or Dataset], destination: Path):
    with open(destination, 'wb') as f:
        pickle.dump(flatten_values(values=values), f)

In [ ]:
save_dataset(resample(omni_data("2024/01/01", "2026/01/01"), 60.*5), Path("omni_5min.pkl"))

In [ ]:
save_dataset(resample(themis_data("2024/01/01", "2026/01/01", "a"), 60.*5), Path("themis_a_5min.pkl"))

In [ ]:
save_dataset(resample(themis_data("2024/01/01", "2026/01/01", "b"), 60.*5), Path("themis_b_5min.pkl"))

In [ ]:
save_dataset(resample(themis_data("2024/01/01", "2026/01/01", "e"), 60.*5), Path("themis_e_5min.pkl"))

In [ ]:
save_dataset(resample(mms1_data("2024/01/01", "2026/01/01"), 60.*5), Path("mms1_5min.pkl"))

In [ ]:
save_dataset(resample(ground_mag_data("2024/01/01", "2026/01/01"), 60.*5), Path("ground_mag_5min.pkl"))

In [7]:
files = glob("*pkl")

In [25]:
time_vector = spz.signal.resampling.generate_time_vector("2024/01/01", "2026/01/01", 60.*5)

In [33]:
df = pd.DataFrame()
for fname in files:
    with open(fname, 'rb') as f:
        dataset=pickle.load(f)
        for v in dataset:
            v=spz.signal.resampling.interpolate(time_vector,v)
            _df = v.to_dataframe()
            if p := v.name:
                _df.columns = [ f"{p}_{l}" for l in _df.columns]
            df = pd.concat([df, _df], axis=1)

In [36]:
df.to_pickle("IMAOC7_summer_school_dataset.pkl")

In [37]:
df.to_csv("IMAOC7_summer_school_dataset.csv")

In [38]:
df.columns

Index(['proton_density_Proton density', 'Pressure_Flow pressure',
       'T_temperature', 'Vx_Vx Velocity, GSE', 'Vy_Vy Velocity, GSE',
       'Vz_Vz Velocity, GSE', 'BX_GSE_Bx, GSE', 'BY_GSE_By, GSE',
       'BZ_GSE_Bz, GSE', 'tha_bs_gsm_bx', 'tha_bs_gsm_by', 'tha_bs_gsm_bz',
       'tha_v_i_vx', 'tha_v_i_vy', 'tha_v_i_vz', 'tha_n_i_ion density',
       'thb_bs_gsm_bx', 'thb_bs_gsm_by', 'thb_bs_gsm_bz', 'thb_v_i_vx',
       'thb_v_i_vy', 'thb_v_i_vz', 'thb_n_i_ion density', 'mms1_b_gse_bx',
       'mms1_b_gse_by', 'mms1_b_gse_bz', 'mms1_dis_vgse_vx',
       'mms1_dis_vgse_vy', 'mms1_dis_vgse_vz', 'mms1_dis_ni_density', 'TAMX',
       'TAMY', 'TAMZ', 'TAMF', 'SOKX', 'SOKY', 'SOKZ', 'SOKF', 'EDAX', 'EDAY',
       'EDAZ', 'EDAF', 'CLFX', 'CLFY', 'CLFZ', 'CLFF', 'KOUX', 'KOUY', 'KOUZ',
       'KOUF', 'IPMX', 'IPMY', 'IPMZ', 'IPMF', 'PPTX', 'PPTY', 'PPTZ', 'PPTF'],
      dtype='str')